In [79]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import re

In [80]:
def combine_excel_sheets(file_path):
    # Load Excel file
    xls = pd.ExcelFile(file_path)
    
    # Get all sheet names
    sheet_names = xls.sheet_names
    
    # Remove "Job Configuration" sheets (case-insensitive)
    sheet_names = [
        s for s in sheet_names
        if "job configuration" not in s.lower()
    ]
    
    all_data = []
    
    # Loop over each sheet
    for sheet in sheet_names:
        sheet_data = pd.read_excel(file_path, sheet_name=sheet)
        
        # Select relevant columns:
        # - Exact columns: ScanArea, TrackId
        # - Columns containing "Repetition"
        repetition_cols = [col for col in sheet_data.columns if "Repetition" in col]
        
        selected_cols = ["ScanArea", "TrackId"] + repetition_cols
        
        # Keep only columns that actually exist (avoids errors)
        selected_cols = [col for col in selected_cols if col in sheet_data.columns]
        
        sheet_data_filtered = sheet_data[selected_cols].copy()
        
        # Add sheet name column
        sheet_data_filtered["sheet_name"] = sheet
        
        # Reorder columns (like dplyr::select)
        cols = ["ScanArea", "TrackId", "sheet_name"] + [
            col for col in sheet_data_filtered.columns
            if col not in ["ScanArea", "TrackId", "sheet_name"]
        ]
        sheet_data_filtered = sheet_data_filtered[cols]
        
        # Append to list
        all_data.append(sheet_data_filtered)
    
    # Combine all sheets into one DataFrame
    combined_data = pd.concat(all_data, ignore_index=True)
    
    return combined_data

In [81]:
#In the data frame "Repetition X" doesn't correspond to the actual time 
#For further analysis it Repetition X must correspond to time 
#This function changes time frame to match my time parameters:
#imaged every 4 hours for the first 20 hours and then imaged every 2 for the next 76
def rename_repetition_columns(df: pd.DataFrame) -> pd.DataFrame:
    #Start time at 0 hours
    current_time = 0
    
    new_columns = []
    #Loop over all Repetition Columns
    for col_name in df.columns:
        #Check if column starts with "Repetition"
        if col_name.startswith("Repetition"):
            #new column name with the current time
            new_col_name = f"Repetition {current_time}"
            new_columns.append(new_col_name)

            #change the time: first 20 hours are 4 hours apart, the rest are 2 hours apart
            if current_time < 20:
                current_time += 4
            else:
                current_time += 2
        else:
            #keep non Repetition names
            new_columns.append(col_name)

    df.columns = new_columns
    return df

In [82]:
#In the data frame "Repetition X" doesn't correspond to the actual time 
#For further analysis it Repetition X must correspond to time 
#This function changes time frame to match my time parameters:
#imaged every hour
def rename_repetition_columns2(df: pd.DataFrame) -> pd.DataFrame:
    #Start time at 0 hours
    current_time = 0
    
    new_columns = []
    #Loop over all "Repetition" Columns
    for col_name in df.columns:
        #Check if column starts with "Repetition"
        if col_name.startswith("Repetition"):
            #new column name with the current time
            new_col_name = f"Repetition {current_time}"
            new_columns.append(new_col_name)

            #change the time: first 20 hours are 4 hours apart, the rest are 2 hours apart
            if current_time < 20:
                current_time += 1
            else:
                current_time += 1
        else:
            #keep non Repetition names
            new_columns.append(col_name)

    df.columns = new_columns
    return df

In [83]:
#In the data frame "Repetition X" doesn't correspond to the actual time 
#For further analysis it Repetition X must correspond to time 
#This function changes time frame to match my time parameters:
#imaged every other hour
def rename_repetition_columns3(df: pd.DataFrame) -> pd.DataFrame:
    #Start time at 0 hours
    current_time = 0
    
    new_columns = []
    #Loop over all Repetition Columns
    for col_name in df.columns:
        #Check if column starts with "Repetition"
        if col_name.startswith("Repetition"):
            #new column name with the current time
            new_col_name = f"Repetition {current_time}"
            new_columns.append(new_col_name)

            #change the time: first 20 hours are 4 hours apart, the rest are 2 hours apart
            if current_time < 20:
                current_time += 2
            else:
                current_time += 2
        else:
            #keep non Repetition names
            new_columns.append(col_name)

    df.columns = new_columns
    return df

In [84]:
#determine the extent of istropic growth 
#imput must be include:
#oCelloScope data frame that is transformed to have data type to be saved under coulumn "sheet_name" and Repetition number corresponds to hour of experiment
#oCelloScope data must also have "TrackId" 
#species, media, and date must also be str. its important for final data sets

#return: isotropic_growth[1] or isotropic_growth[data] is a data frame
#return: isotropic_growth[2] or isotropic_growth[plot] is a plot showing segmenteted regression

def isotropic_growth(df, species, media, date):

    # Find repetition columns
    time_columns = [col for col in df.columns if "Repetition" in col]

    # Pivot to long format
    data_long = df.melt(
        id_vars=[col for col in df.columns if col not in time_columns],
        value_vars=time_columns,
        var_name="time",
        value_name="value"
    )

    # Convert repetition column names to numeric time
    data_long["time"] = (
        data_long["time"]
        .str.replace(r"[^0-9.]", "", regex=True)
        .astype(float)
    )

    # Filter for area data only
    area_data = data_long[data_long["sheet_name"] == "Area (um2)"].copy()

    # Remove ungerminated conidia
    # Keep only conidia that grew to at least 2.5x their original size
    area_data["Original_Size"] = area_data.groupby("TrackId")["value"].transform("first")
    area_data["Max_Size"] = area_data.groupby("TrackId")["value"].transform("max")
    
    
    #filter only area data
    area_data_filtered = area_data[
        area_data["Max_Size"] > 2.5 * area_data["Original_Size"]
    ].copy()

    #stall code if no area over no 2.5X (look at original video to confirm germination)
    if area_data_filtered.empty:
        raise ValueError("No rows remain after filtering for growth > 2.5x original size.")

    # Prepare data for segmented regression by removing excess data
    model_df = area_data_filtered[["time", "value"]].dropna().copy()

    #confirm there is enough time period points to make an accurate segmented regression 
    if len(model_df) < 4:
        raise ValueError("Not enough data points for segmented regression.")

    #confirm only individual data points and individual conidia are viewed
    unique_times = np.sort(model_df["time"].unique())

    #list of potential breakpoints
    candidate_breaks = unique_times[1:-1]

    #name initial values
    best_break = None
    best_model = None
    best_rss = np.inf

    # Piecewise linear model with one breakpoint:
    # y = b0 + b1*time + b2*max(0, time - breakpoint)
    for bp in candidate_breaks:
        x = model_df["time"].values
        hinge = np.maximum(0, x - bp)

        X = pd.DataFrame({
            "intercept": 1.0,
            "time": x,
            "hinge": hinge
        })

        fit = sm.OLS(model_df["value"].values, X).fit()
        rss = np.sum(fit.resid ** 2)
        
        #looping through and confirmining best fit for all possible timepoints
        if rss < best_rss:
            best_rss = rss
            best_break = bp
            best_model = fit
    
    #pullout best fit breakpoint
    breakpoints = np.array([best_break])

    # Predicted area at breakpoint using model deirved above
    bp_X = pd.DataFrame({
        "intercept": [1.0],
        "time": [best_break],
        "hinge": [0.0]
    })
    breakpoint_values = best_model.predict(bp_X).values

    #Pull inital area of conidia at first recorded time
    first_recording_time = area_data_filtered["time"].min()
    first_area = area_data_filtered.loc[
        area_data_filtered["time"] == first_recording_time, "value"
    ].mean()

    # Results dataframe of all conidia
    breakpoint_results = pd.DataFrame({
        "Species": [species],
        "Media": [media],
        "Date": [date],
        "Breakpoint_Time": [breakpoints[0]],
        "Breakpoint_Area": [breakpoint_values[0]],
        "Average_First_Area": [first_area],
        #Simple fold change
        "Swelling_change": [breakpoint_values[0] / first_area]
    })

    #Add predictions for visualization
    #pull time values
    x_all = area_data_filtered["time"].values

    #derive hinge point (breakpoint) in model
    hinge_all = np.maximum(0, x_all - best_break)

    pred_X = pd.DataFrame({
        "intercept": 1.0,
        "time": x_all,
        "hinge": hinge_all
    })

    model_data = area_data_filtered.copy()
    model_data["predicted"] = best_model.predict(pred_X)

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(8, 6))

    sns.scatterplot(data=model_data, x="time", y="value", ax=ax)
    sns.lineplot(data=model_data.sort_values("time"), x="time", y="predicted", ax=ax)

    ax.axvline(x=best_break, linestyle="--")
    ax.set_title(f"Segmented Regression with Breakpoints and Values - {species}")
    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel("Area (μm²)")

    plt.tight_layout()

    return {
        "data": breakpoint_results,
        "plot": fig
    }

In [85]:
def find_time_point(df, breakpoint_value, species, media, date):
    # Get time point columns containing "Repetition"
    time_cols = [col for col in df.columns if "Repetition" in col]

    # Unique TrackIds
    track_ids = pd.unique(df["TrackId"])

    # Initialize output lists
    selected_time_points = [np.nan] * len(track_ids)
    last_time_points = [np.nan] * len(track_ids)
    selected_lengths = [np.nan] * len(track_ids)
    final_lengths = [np.nan] * len(track_ids)

    confirm_n = 3

    # Loop through each TrackId
    for i, track_id in enumerate(track_ids):
        track_data = df[df["TrackId"] == track_id]

        # Separate Circularity, Area, and TotalLength data
        circularity_data = track_data[track_data["sheet_name"] == "Circularity"]
        area_data = track_data[track_data["sheet_name"] == "Area (um2)"]
        length_data = track_data[track_data["sheet_name"] == "TotalLength (um)"]

        # Skip if any required sheet is missing
        if circularity_data.empty or area_data.empty or length_data.empty:
            continue

        # Use first row if multiple rows exist for a TrackId/sheet_name combination
        circularity_row = circularity_data[time_cols].iloc[0]
        area_row = area_data[time_cols].iloc[0]
        length_row = length_data[time_cols].iloc[0]

        # Find first valid initial area: non-NA and between 10 and 75
        initial_area = np.nan
        for val in area_row:
            if pd.notna(val) and 10 <= val <= 75:
                initial_area = val
                break

        # This variable exists in your R code but is not used later
        initial_circle = np.nan
        for val in circularity_row:
            if pd.notna(val) and val < 0.92:
                initial_circle = val
                break

        # Find germination time point
        if pd.notna(initial_area):
            for j, col in enumerate(time_cols):
                current_circularity = circularity_row[col]
                current_area = area_row[col]

                if pd.notna(current_circularity) and pd.notna(current_area):
                    if current_circularity < 0.94 and current_area >= (breakpoint_value * initial_area * 1.25):
                        L = len(time_cols)
                        confirmed = False

                        if j + confirm_n < L:
                            confirmed = True
                            for k in range(j + 1, j + confirm_n + 1):
                                next_circ = circularity_row[time_cols[k]]
                                next_area = area_row[time_cols[k]]

                                if (
                                    pd.isna(next_circ)
                                    or pd.isna(next_area)
                                    or not (
                                        next_circ < 0.94
                                        and next_area >= (breakpoint_value * initial_area * 1.25)
                                    )
                                ):
                                    confirmed = False
                                    break
                        else:
                            # Accept when fewer than confirm_n future frames exist
                            confirmed = True

                        if not confirmed:
                            continue

                        selected_time_points[i] = col
                        selected_lengths[i] = length_row[col]
                        break

        # Find last non-NA length and corresponding time point
        final_length_found = np.nan
        for col in time_cols:
            if pd.notna(length_row[col]):
                final_length_found = length_row[col]
                last_time_points[i] = col

        final_lengths[i] = final_length_found

    # Helper to extract numeric part from strings like "Repetition 12"
    def extract_numeric(value):
        if pd.isna(value):
            return np.nan
        nums = re.sub(r"[^0-9.]", "", str(value))
        return float(nums) if nums else np.nan

    # Build ScanArea map per TrackId
    scanarea_map = (
        df[["TrackId", "ScanArea"]]
        .drop_duplicates(subset=["TrackId"])
        .set_index("TrackId")["ScanArea"]
    )

    result = pd.DataFrame({
        "ScanArea": [scanarea_map.get(track_id, np.nan) for track_id in track_ids],
        "TrackId": track_ids,
        "selected_time_point": [extract_numeric(x) for x in selected_time_points],
        "selected_time_point_word": selected_time_points,
        "last_time_point": [extract_numeric(x) for x in last_time_points],
        "selected_length": selected_lengths,
        "final_length": final_lengths,
    }).drop_duplicates()

    # Add metadata and growth rates
    result["Species"] = species
    result["Media"] = media
    result["Date"] = date

    result["linear_growth_rate"] = (
        (result["final_length"] - result["selected_length"]) /
        (result["last_time_point"] - result["selected_time_point"])
    )

    result["exponential_growth_rate"] = (
        (np.log(result["final_length"]) - np.log(result["selected_length"])) /
        (result["last_time_point"] - result["selected_time_point"])
    )

    # Reorder columns to match the R output
    result = result[
        [
            "Species",
            "Media",
            "Date",
            "ScanArea",
            "TrackId",
            "selected_time_point",
            "selected_time_point_word",
            "last_time_point",
            "selected_length",
            "final_length",
            "linear_growth_rate",
            "exponential_growth_rate",
        ]
    ]

    return result

In [86]:
def length_post_germination(df, germination_results, species, media, date):
    species_name = species
    media_type = media
    date_of_experiment = date

    # Get time columns
    time_cols = [col for col in df.columns if "Repetition" in col]

    # Extract numeric time values from column names
    def extract_time(col):
        m = re.search(r"([0-9]+\.?[0-9]*)", str(col))
        return float(m.group(1)) if m else np.nan

    time_intervals = np.array([extract_time(col) for col in time_cols], dtype=float)

    # Filter for TotalLength data and merge with germination results
    df_germinated = (
        df[df["sheet_name"] == "TotalLength (um)"]
        .merge(germination_results, on="TrackId", how="inner")
        .copy()
    )

    relative_time_data_list = []

    # Loop through each germinated row
    for _, row in df_germinated.iterrows():
        track_id = row["TrackId"]
        germination_time_point = row["selected_time_point_word"]

        sample_data = df[
            (df["TrackId"] == track_id) &
            (df["sheet_name"] == "TotalLength (um)")
        ]

        if sample_data.empty:
            continue

        # Take first matching row
        sample_row = sample_data.iloc[0]

        length_values = pd.to_numeric(sample_row[time_cols], errors="coerce").to_numpy()

        if np.all(np.isnan(length_values)):
            continue

        # Find germination index
        if germination_time_point not in time_cols:
            continue

        germination_index = time_cols.index(germination_time_point)

        if germination_index >= len(length_values):
            continue

        relative_times = time_intervals - time_intervals[germination_index]

        relative_time_data = pd.DataFrame({
            "track_id": track_id,
            "relative_time": relative_times,
            "length": length_values
        })

        relative_time_data_list.append(relative_time_data)

    # Combine all samples
    if relative_time_data_list:
        relative_time_data_combined = pd.concat(relative_time_data_list, ignore_index=True)
    else:
        relative_time_data_combined = pd.DataFrame(columns=["track_id", "relative_time", "length"])

    # Remove missing lengths
    relative_time_data_combined = relative_time_data_combined.dropna(subset=["length"]).copy()

    # Median summary
    if not relative_time_data_combined.empty:
        total_length_median = (
            relative_time_data_combined
            .groupby("relative_time", as_index=False)
            .agg(
                Median_Length=("length", "median"),
                n=("length", "count")
            )
        )
        total_length_median["log_Median_Length"] = np.log(total_length_median["Median_Length"])
        total_length_median["Species"] = species_name
        total_length_median["Media"] = media_type
        total_length_median["Date"] = date_of_experiment
    else:
        total_length_median = pd.DataFrame(
            columns=[
                "relative_time", "Median_Length", "log_Median_Length",
                "n", "Species", "Media", "Date"
            ]
        )

    # Linear model on 0 <= relative_time <= 24
    fit_subset = total_length_median[
        (total_length_median["relative_time"] >= 0) &
        (total_length_median["relative_time"] <= 24)
    ].copy()

    total_length_median["lm_predicted"] = np.nan
    total_length_median["exm_predicted"] = np.nan

    if len(fit_subset) >= 2:
        # Linear model
        lm_model = smf.ols("Median_Length ~ relative_time", data=fit_subset).fit()
        total_length_median["lm_predicted"] = lm_model.predict(total_length_median)

        # Exponential model via log transform
        exm_subset = fit_subset[fit_subset["Median_Length"] > 0].copy()
        if len(exm_subset) >= 2:
            exm_model = smf.ols("log_Median_Length ~ relative_time", data=exm_subset).fit()
            total_length_median["exm_predicted"] = np.exp(exm_model.predict(total_length_median))

    # y-axis limit
    valid_window = relative_time_data_combined[
        (relative_time_data_combined["relative_time"] >= 0) &
        (relative_time_data_combined["relative_time"] <= 24)
    ]

    if not valid_window.empty:
        y_limit = valid_window["length"].max() + 10
    else:
        y_limit = None

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Individual conidia lines
    if not relative_time_data_combined.empty:
        for track_id, group in relative_time_data_combined.groupby("track_id"):
            group = group.sort_values("relative_time")
            ax.plot(
                group["relative_time"],
                group["length"],
                color="black",
                alpha=0.7,
                linewidth=1
            )
            ax.scatter(
                group["relative_time"],
                group["length"],
                color="black",
                s=15,
                alpha=0.7
            )

    # Median line
    if not total_length_median.empty:
        ax.plot(
            total_length_median["relative_time"],
            total_length_median["Median_Length"],
            linewidth=2,
            label="Median Length"
        )

    # Linear model
    if "lm_predicted" in total_length_median.columns and total_length_median["lm_predicted"].notna().any():
        ax.plot(
            total_length_median["relative_time"],
            total_length_median["lm_predicted"],
            linewidth=2,
            label="Linear Growth Model"
        )

    # Exponential model
    if "exm_predicted" in total_length_median.columns and total_length_median["exm_predicted"].notna().any():
        ax.plot(
            total_length_median["relative_time"],
            total_length_median["exm_predicted"],
            linewidth=2,
            label="Exponential Growth Model"
        )

    ax.set_title("Length Over Time Post-Germination (Germination Time = 0)")
    ax.set_xlabel("Relative Time since Germination (Hours)")
    ax.set_ylabel("Length (µm)")
    ax.set_xlim(0, 24)
    if y_limit is not None:
        ax.set_ylim(0, y_limit)
    ax.legend()
    ax.grid(True, alpha=0.3)

    return {
        "relative_time_df": relative_time_data_combined,
        "median_length_df": total_length_median,
        "plot": fig
    }